# 02 — Customer segmentation (RFM + k-means)

**Questions I'm trying to answer**
- Can we split registered customers into a few useful groups with just Recency / Frequency / Monetary?
- Does silhouette prefer k=3, 4, 5… or is k=4 "good enough" to label?

Registered only (null CustomerID rows already dropped upstream).


In [ ]:
from pathlib import Path
import pandas as pd, json

ROOT = Path("..")
seg = pd.read_csv(ROOT / "data" / "gold" / "customer_segments.csv")
summary = pd.read_csv(ROOT / "python" / "outputs" / "segment_summary.csv")
meta = json.loads((ROOT / "reports" / "segmentation_metrics.json").read_text())

print("k =", meta["k"], "  silhouette =", meta["silhouette_full"], "  n =", meta["n_customers"])
print("segments:", [s["Segment"] for s in meta["segments"]])
summary[["Segment", "Customers", "RecencyDays", "Frequency", "Monetary"]].round(1)

### Quick check
Do Champions look like low recency + high monetary? Do At Risk / Lapsed look like the opposite?

If a refresh flips the names, re-check the labeling rules in `python/02_customer_segmentation.py`.


In [ ]:
# median RFM by segment — does the naming still make sense?
seg.groupby("Segment")[["RecencyDays", "Frequency", "Monetary"]].median().round(1)

In [ ]:
# how much revenue sits in each segment?
rev = seg.groupby("Segment")["Monetary"].sum().sort_values(ascending=False)
print((rev / rev.sum()).map(lambda x: f"{x:.1%}"))
rev

**Takeaway (for me):** a small Champions group usually carries a big share of revenue — useful when drafting a retention list, even if the silhouette isn't amazing (~0.36).
